In [55]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [56]:
artists = pd.read_json('artists.jsonl', lines=True)
genres_to_filter = ['pop', 'trap', 'rap', 'rock', 'metal', 'hip hop']

In [57]:
def filter_genres(genre_list, regex_pattern):
    return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]


In [58]:
filtered_artists = pd.DataFrame()
filtered_genres_dict = {}
for genre in genres_to_filter:
    regex_pattern = fr'\b({genre})\b'

    artists['filtered_genres'] = artists['genres'].apply(lambda x: filter_genres(x, regex_pattern))
    one_genre = artists[artists['filtered_genres'].apply(lambda x: x != [])]
    one_genre['genres'] = genre
    filtered_artists = pd.concat([filtered_artists, one_genre], ignore_index=True)

# Drop the temporary 'filtered_genres' column
filtered_artists = filtered_artists.drop(columns=['filtered_genres'])

# Display the concatenated DataFrame
print(filtered_artists)

/tmp/ipykernel_728/77786420.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]
/tmp/ipykernel_728/395593341.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  one_genre['genres'] = genre
/tmp/ipykernel_728/77786420.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]


                          id                name   genres
0     7jVv8c5Fj3E9VhNjxT4snq           Lil Nas X      pop
1     0xRXCcSX89eobfrshSVdyu              MEDUZA      pop
2     4f7KfxeHq9BiylGmyXepGt      Tanishk Bagchi      pop
3     56mfhUDKa1vec6rSLZV5Eg           Jawsh 685      pop
4     6m6e7D2TnV0aYMllFFwMxu  Guilherme & Benuto      pop
...                      ...                 ...      ...
2277  50co4Is1HCEo8bhOyUWKpn          Young Thug  hip hop
2278  6U3ybJ9UHNKEdsH7ktGBZ7                 JID  hip hop
2279  1qQLhymHXFPtP5U8KNKsm6          Apache 207  hip hop
2280  6qgnBH6iDM91ipVXv28OMu          KAYTRANADA  hip hop
2281  46MWeeHNVMYRIIofQBEX98                BEAM  hip hop

[2282 rows x 3 columns]


In [59]:
len([ x for x in filtered_artists['genres'] if x == 'trap'])

268

In [60]:
filtered_artists.describe()

,id,name,genres
count,2282,2282,2282
unique,1391,1391,6
top,5gCRApTajqwbnHHPbr2Fpi,Juicy J,pop
freq,4,4,919


In [62]:
tracks = pd.read_json('tracks.jsonl', lines=True)
tracks.sample()

,id,name,popularity,duration_ms,explicit,id_artist,release_date,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
15084,6oC8fM5FcQy8Hpw7oQpKEy,Tras Vos,57,170500,0,2UZIAOlrnyZmyzt1nuXr9y,2019-11-08,0.836,0.578,11,-7.101,0.107,0.26,0.116,0.11,0.415,94.006


In [79]:
divided_songs = {}
for genre in genres_to_filter:
    genre_frame = filtered_artists[filtered_artists['genres'] == genre]
    genre_artist_ids = genre_frame['id'].tolist()
    divided_songs[genre] = tracks[tracks['id_artist'].apply(lambda x: x in genre_artist_ids)]

In [81]:
divided_songs['pop']

,id,name,popularity,duration_ms,explicit,id_artist,release_date,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
29,1XPj5quoeV5Gd0paSUDpvm,You Send Me - Mono,61,165733,0,6hnWRPzGGKiapVX1UCdEAC,1958-02-01,0.572,0.365,4,-7.583,0.0276,0.8800,0.00000,0.1250,0.4100,96.022
45,2G0GextMwZJLkNxcSZ7ZJ3,(What A) Wonderful World - Mono,67,128787,0,6hnWRPzGGKiapVX1UCdEAC,1960-02,0.686,0.672,11,-5.523,0.0323,0.7000,0.00000,0.1350,0.8570,128.550
46,4yjz1aazw6R8ZURpGbCAkp,Bring It On Home To Me,65,165280,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.482,0.569,0,-5.897,0.0328,0.6170,0.00000,0.1550,0.6090,70.843
48,0BFEyqJ9DJXS7gKg0Kj46R,You Send Me,64,165560,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.553,0.291,4,-10.426,0.0301,0.8780,0.00000,0.1290,0.4070,96.217
50,5Fih0qlqrBjPIE0dFamuBr,(What A) Wonderful World,62,129293,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.763,0.571,11,-10.552,0.0337,0.5290,0.00000,0.1540,0.9360,128.211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22407,0LcNMuOiULmxJK3bdHTfDF,Missing You,65,214987,0,7gAppWoH7pcYmphCVTXkzs,2019-04-19,0.537,0.647,2,-7.370,0.0519,0.1680,0.00000,0.1080,0.5310,79.770
22408,1uviKYHZuM4uINK33F7sCt,Fix It to Break It,70,198799,0,7okSU80WTrn4LXlyXYbX3P,2020-03-27,0.493,0.461,2,-8.524,0.0456,0.8450,0.00000,0.1150,0.3500,51.414
22409,1fXmDeiCb3ABt5CzkMxp4u,Lotus Inn,63,195868,0,2jnIB6XdLvnJUeNTy5A0J2,2020-12-04,0.578,0.740,2,-5.080,0.0721,0.0119,0.00000,0.1800,0.4950,155.984
22410,27kcZEJvhkb1rzZS9gCpdA,remember the mornings,67,202355,0,7okSU80WTrn4LXlyXYbX3P,2020-11-27,0.590,0.443,0,-5.662,0.0722,0.6180,0.00000,0.1100,0.3420,149.447
